In [157]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from torch.utils.data import Dataset, random_split
import torch.nn as nn


import scienceplots
plt.style.use(['science','notebook'])

In [158]:
def plot(x):
    plt.imshow(x)
    plt.axis('auto')
    plt.show()

In [159]:
n = 64
data = np.zeros(shape=(n**2, 3))

def operation(i,j, n, op):
    if op == "mod_n":
        return (i + j) % n 
    if op == "bitwise_and":
        return (i & j)

for i in range(n):
    for j in range(n):
        data[(i *n) + j] = [i, j, operation(i,j, n, "bitwise_and")]

In [160]:
class GrokkingDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = torch.from_numpy(dataset).long()

    def __len__(self):
        return self.dataset.shape[0]

    def __getitem__(self, idx):
        return self.dataset[idx, :2], self.dataset[idx, 2]

In [162]:
dataset = GrokkingDataset(data)
train_indices, test_indices = random_split(dataset, [0.4, 0.6])

x_train, y_train = dataset[train_indices.indices]
x_test, y_test = dataset[test_indices.indices]


In [174]:
a = torch.tensor([1,2])
b = torch.tensor([2,3])

torch.cat([a,b])


tensor([1, 2, 2, 3])

In [163]:
class GrokkingModel(nn.Module):
    def __init__(self, embed_dimension, hidden_dimension, n):
        super().__init__()

        self.left_embed = nn.Embedding(n, embed_dimension)
        self.right_embed = nn.Embedding(n, embed_dimension)
        self.hidden = nn.Linear(2*embed_dimension, hidden_dimension, bias=False)
        self.relu = nn.ReLU()
        self.unembedding = nn.Linear(hidden_dimension, n, bias=False)


    def forward(self, input):
        a_embed = self.left_embed(input[:, 0])
        b_embed = self.right_embed(input[:, 1])

        x = torch.cat((a_embed,b_embed), dim=1)
        x = self.hidden(x)
        x = self.relu(x)
        logits = self.unembedding(x)
        return logits

In [164]:
model = GrokkingModel(128, 6, n)

In [ ]:
epochs = 10_000


optimizer = torch.optim.AdamW(model.parameters(), weight_decay=1)
loss_func = nn.CrossEntropyLoss()

training_losses, test_losses = [], []


for i in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = loss_func(logits, y_train)
    loss.backward()
    optimizer.step()
    
    if i % 100 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(x_test)
            test_loss = loss_func(test_logits, y_test)
  
        training_losses.append(loss.item())
        test_losses.append(test_loss.item())

        print(f"Epoch {i} | Loss {loss} | Test loss {test_loss}")

        if test_loss < 0.00000001:
            break


Epoch 0 | Loss 4.2226362228393555 | Test loss 4.207289695739746
Epoch 100 | Loss 3.142122268676758 | Test loss 3.388289451599121
Epoch 200 | Loss 2.112098217010498 | Test loss 2.4159293174743652
Epoch 300 | Loss 1.5566790103912354 | Test loss 1.8398810625076294
Epoch 400 | Loss 1.1158024072647095 | Test loss 1.3643079996109009
Epoch 500 | Loss 0.8045132160186768 | Test loss 1.0219401121139526
Epoch 600 | Loss 0.5926114320755005 | Test loss 0.8007003664970398
Epoch 700 | Loss 0.42690524458885193 | Test loss 0.6257657408714294
Epoch 800 | Loss 0.30126750469207764 | Test loss 0.4691559076309204
Epoch 900 | Loss 0.22301806509494781 | Test loss 0.3610568344593048
Epoch 1000 | Loss 0.16634413599967957 | Test loss 0.287489116191864
Epoch 1100 | Loss 0.12888921797275543 | Test loss 0.2415710687637329
Epoch 1200 | Loss 0.1093650758266449 | Test loss 0.21646757423877716
Epoch 1300 | Loss 0.09811898320913315 | Test loss 0.20116879045963287
Epoch 1400 | Loss 0.09021933376789093 | Test loss 0.18987

KeyboardInterrupt: 

In [166]:
left = model.left_embed.weight
right = model.right_embed.weight
hidden = model.hidden.weight


left = model.left_embed.shape

AttributeError: 'Embedding' object has no attribute 'shape'